# v2/v3: Severstal → NEU-DET transfer learning

Run with **Runtime → Run all**. The notebook:

1. Clones the repo and restores earlier results from Google Drive
2. Builds the NEU-DET and Severstal datasets
3. Trains the **v1r** control, **v2** Severstal pretraining, and the **v3 / v3b** fine-tunes
4. Evaluates every version on the NEU-DET test split and prints a comparison

Plan: [`docs/SEVERSTAL_TRANSFER_PLAN.md`](../docs/SEVERSTAL_TRANSFER_PLAN.md)

## One-time setup
1. **Runtime → Change runtime type → T4 GPU**
2. Accept the competition rules: https://www.kaggle.com/competitions/severstal-steel-defect-detection/rules
3. Kaggle → Settings → **Create New Token** (downloads `kaggle.json`)
4. In Colab, open the 🔑 **Secrets** panel and add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from `kaggle.json`), with notebook access enabled

## If the session disconnects
Run all again. Finished versions are skipped, and interrupted training resumes from its last checkpoint, which is saved on Drive.

In [ ]:
# ---- Settings ----
REPO_URL = "https://github.com/DEEPIKAGRACE22/PS-7---AI-Assisted-Quality-Inspection-IBM-x-MAH-Quantum-project-.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/ps7_severstal"   # checkpoints and results persist here

SEEDS = [0, 1, 2]          # seeds for v1r / v3 / v3b (use [0] for a quick first pass)
RUN_CONTROL = True         # v1r: NEU-only baseline with the same schedule as v3
RUN_PRETRAIN = True        # v2:  Severstal pretraining (~3-5 h on a T4)
RUN_FINETUNE = True        # v3:  fine-tune from v2
RUN_FINETUNE_FROZEN = True # v3b: fine-tune from v2 with frozen backbone

In [ ]:
import os, subprocess, sys
from pathlib import Path

import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive, userdata
drive.mount("/content/drive")

REPO = Path("/content/ps7")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [ ]:
# Keep training runs on Drive so checkpoints survive disconnects,
# and restore any model versions registered in earlier sessions.
drive_dir = Path(DRIVE_DIR)
(drive_dir / "runs").mkdir(parents=True, exist_ok=True)
(drive_dir / "models").mkdir(parents=True, exist_ok=True)

if not Path("runs").is_symlink():
    subprocess.run(["rm", "-rf", "runs"])
    Path("runs").symlink_to(drive_dir / "runs")

subprocess.run(["cp", "-rn", f"{drive_dir}/models/.", "models/"], check=True)

def run(*cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")

def save_to_drive():
    subprocess.run(["cp", "-r", "models/.", f"{drive_dir}/models/"], check=True)

def trained(version):
    return Path(f"models/{version}/weights/best.pt").exists()

print("Registered versions:", sorted(p.name for p in Path("models").iterdir() if p.is_dir()))

## 1. Datasets

In [ ]:
run(sys.executable, "scripts/prepare_neu_det.py")

In [ ]:
# Severstal is only needed until v2 is trained
NEED_SEVERSTAL = RUN_PRETRAIN and not trained("v2_yolo11n_severstal_pretrain")

if NEED_SEVERSTAL:
    run(sys.executable, "scripts/prepare_severstal.py")
else:
    print("v2 already trained, skipping Severstal preparation.")

### Sanity check
The boxes must sit on visible defects. If they look wrong (shifted, transposed, or huge merged regions), stop here and fix `prepare_severstal.py` before training.

In [ ]:
from IPython.display import Image, display

run(sys.executable, "scripts/visualize_dataset.py", "data/neu_yolo", "--split", "train", "--n", "8")
display(Image("data/neu_yolo/preview_train.png"))

if NEED_SEVERSTAL:
    run(sys.executable, "scripts/visualize_dataset.py", "data/severstal_yolo",
        "--split", "train", "--n", "16", "--defects-only")
    display(Image("data/severstal_yolo/preview_train.png"))

## 2. Training

In [ ]:
if RUN_CONTROL:
    for seed in SEEDS:
        run(sys.executable, "scripts/train.py", "configs/train/v1r_yolo11n_baseline_long.yaml",
            "--seed", seed, "--skip-existing")
        save_to_drive()

In [ ]:
if RUN_PRETRAIN:
    run(sys.executable, "scripts/train.py", "configs/train/v2_yolo11n_severstal_pretrain.yaml", "--skip-existing")
    save_to_drive()

In [ ]:
finetune_configs = []
if RUN_FINETUNE:
    finetune_configs.append("configs/train/v3_yolo11n_severstal_ft.yaml")
if RUN_FINETUNE_FROZEN:
    finetune_configs.append("configs/train/v3b_yolo11n_severstal_ft_freeze.yaml")

if finetune_configs:
    assert trained("v2_yolo11n_severstal_pretrain"), "v2 pretraining must finish before fine-tuning."

for config in finetune_configs:
    for seed in SEEDS:
        run(sys.executable, "scripts/train.py", config, "--seed", seed, "--skip-existing")
        save_to_drive()

## 3. Evaluation on the NEU-DET test split

In [ ]:
neu_versions = sorted(
    p.parent.parent.name for p in Path("models").glob("*/weights/best.pt")
    if not p.parent.parent.name.startswith("v2_")   # v2 has Severstal classes, not NEU
)
for version in neu_versions:
    if not Path(f"models/{version}/metrics/test_per_class.csv").exists():
        run(sys.executable, "scripts/evaluate.py", version)

save_to_drive()
run(sys.executable, "scripts/compare.py")

## 4. Bring results back to the repo

Download the zip, unzip it into the repo's `models/` folder, add rows to `models/README.md`, and commit. The weights are about 5 MB each.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/ps7_models", "zip", "models")
print("Also saved on Drive:", drive_dir / "models")
files.download(archive)